In [1]:
# ================================================================
#  Notebook : Analyse des activités sportives et scolaires en France
#  Objectif : Aider un couple de retraités à choisir une zone
#               avec bonne qualité de vie, activités sportives,
#               et écoles pour leurs petits-enfants.
#  Version optimisée : Filtre en amont et widgets pertinents
# ================================================================

# --- 1. Import des bibliothèques ---
import pandas as pd
import geopandas as gpd
import folium
import warnings
from folium.plugins import MarkerCluster
from ipywidgets import interact, Dropdown, Checkbox, SelectMultiple, Button, VBox, HBox, Output, interactive_output
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from shapely.geometry import Point

# Configuration pour Matplotlib dans le notebook
%matplotlib inline

# ================================================================
# 2️ Chargement des fichiers
# ================================================================

print("Chargement des données...")

# --- Équipements sportifs ---
# On spécifie les colonnes utiles pour optimiser la mémoire
cols_equip = ['equip_numero', 'inst_nom', 'new_name', 'dep_code', 'dep_nom', 'reg_nom', 
              'equip_nom', 'equip_type_name', 'equip_y', 'equip_x']
equip_df = pd.read_csv("data-es.csv", sep=";", encoding="utf-8", low_memory=False, usecols=cols_equip)

# --- Annuaire éducation ---
cols_edu_reels = [
    'Identifiant_de_l_etablissement', 
    'Nom_etablissement', 
    'Nom_commune', 
    'Libelle_departement',
    'Libelle_region',
    'latitude',
    'longitude'
]
edu_df = pd.read_csv("fr-en-annuaire-education.csv", sep=";", encoding="utf-8-sig", 
                   low_memory=False, on_bad_lines='skip', usecols=cols_edu_reels)

# On renomme les colonnes pour qu'elles correspondent au reste du script
map_renommage = {
    'Libelle_departement': 'Nom_departement',
    'Libelle_region': 'Nom_region',
    'latitude': 'Latitude',
    'longitude': 'Longitude'
}
edu_df = edu_df.rename(columns=map_renommage)

print(f"Équipements sportifs (brut) : {len(equip_df):,} lignes")
print(f"Établissements scolaires (brut) : {len(edu_df):,} lignes")

# ================================================================
# 3️ Préparation et FILTRAGE
# ================================================================
print("Nettoyage et filtrage...")

# --- 3.1. Nettoyage et filtrage des équipements sportifs ---

# 1. Supprimer les lignes sans coordonnées ou infos vitales
equip_df = equip_df.dropna(subset=['equip_y', 'equip_x', 'reg_nom', 'dep_nom', 'equip_type_name'])

# 2. 🔥 FILTRE 1 : Exclure l'Île-de-France (le persona veut en partir)
equip_df = equip_df[equip_df['reg_nom'] != 'Île-de-France'].copy()

# 3. 🔥 FILTRE 2 : Se concentrer sur les activités pertinentes pour le persona (retraités)
activites_pertinentes = [
    'Piscine', 'Tennis', 'Golf', 'Boulodrome', 'Pétanque', 'Randonnée', 
    'Gymnastique', 'Yoga', 'Danse', 'Club house', 'Sentier', 'Marche'
]
# On utilise str.contains pour une recherche large (ex: 'Tennis' capture 'Court de tennis')
equip_seniors_df = equip_df[
    equip_df['equip_type_name'].str.contains('|'.join(activites_pertinentes), case=False, na=False)
].copy()

# 4. Créer le GeoDataFrame APRES avoir filtré (beaucoup plus léger)
equip_seniors_gdf = gpd.GeoDataFrame(
    equip_seniors_df,
    geometry=gpd.points_from_xy(equip_seniors_df['equip_x'], equip_seniors_df['equip_y']),
    crs="EPSG:4326"
)

print(f"Équipements sportifs (après filtrage IDF + seniors) : {len(equip_seniors_gdf):,} lignes")

# --- 3.2. Nettoyage et filtrage des établissements scolaires ---

# 1. Supprimer les lignes sans coordonnées ou infos vitales
edu_df = edu_df.dropna(subset=['Latitude', 'Longitude', 'Nom_departement', 'Nom_region'])

# 2. 🔥 FILTRE 1 : Exclure l'Île-de-France
edu_df = edu_df[edu_df['Nom_region'] != 'Île-de-France'].copy()

# 3. Créer le GeoDataFrame à partir du CSV
edu_gdf = gpd.GeoDataFrame(
    edu_df,
    geometry=gpd.points_from_xy(edu_df['Longitude'], edu_df['Latitude']),
    crs="EPSG:4326"
)

print(f"Établissements scolaires (après filtrage IDF) : {len(edu_gdf):,} lignes")
print("Préparation terminée.")

# ================================================================
# 4️ Widget 1 : Classement des Départements (pour choisir OÙ regarder)
# ================================================================
print("\n--- Visualisation 1 : Classement des Départements ---")

# --- Définition des widgets ---
# Liste des activités uniques (basée sur nos données filtrées)
types_activites_options = sorted(equip_seniors_gdf['equip_type_name'].unique())

# On prend dynamiquement les 3 premières options comme valeurs par défaut
# pour éviter une erreur si les valeurs par défaut "en dur" n'existent pas.
if len(types_activites_options) >= 3:
    default_activities = list(types_activites_options[:3])
elif len(types_activites_options) > 0:
    default_activities = list(types_activites_options)
else:
    default_activities = []

activites_widget = SelectMultiple(
    options=types_activites_options,
    value=default_activities,
    description='Activités:',
    layout={'width': '400px', 'height': '150px'}
)

button_viz1 = Button(
    description="🏆 Classer les départements"
)

output_viz1 = Output()

# --- Fonction de mise à jour ---
def update_viz1(b):
    with output_viz1:
        output_viz1.clear_output(wait=True) # Nettoie le graphique précédent
        
        if not activites_widget.value:
            print("Veuillez sélectionner au moins une activité.")
            return

        # Filtrer le dataframe principal sur la base de la sélection
        filtered_df = equip_seniors_gdf[
            equip_seniors_gdf['equip_type_name'].isin(activites_widget.value)
        ]
        
        # Agréger par département
        ranking = filtered_df.groupby('dep_nom')['equip_numero'].count().sort_values(ascending=False).head(20)
        
        if ranking.empty:
            print("Aucun département trouvé pour ces activités.")
            return

        # Afficher le graphique
        plt.figure(figsize=(10, 7))
        ranking.sort_values().plot(kind='barh', color='skyblue')
        plt.title(f"Top 20 des départements (hors IDF) pour les activités choisies")
        plt.xlabel("Nombre d'équipements")
        plt.ylabel("Département")
        plt.tight_layout()
        plt.show()

# --- Liaison et affichage ---
button_viz1.on_click(update_viz1)
viz1_container = VBox([
    HBox([activites_widget, button_viz1]), 
    output_viz1
])
display(viz1_container)


# ================================================================
# 5️ Widgets 2 & 3 : Dashboard d'Exploration Départementale
# (Pour analyser le département choisi)
# ================================================================
print("\n--- Visualisation 2 & 3 : Dashboard d'Exploration ---")

# --- Définition des widgets ---
dep_list = sorted(equip_seniors_gdf['dep_nom'].unique())
dep_widget = Dropdown(
    options=dep_list,
    value=dep_list[0], # Sélectionne le premier de la liste
    description='Département:',
    layout={'width': '400px'}
)

school_widget = Checkbox(
    value=False,
    description="Afficher les écoles (pour petits-enfants)"
)

output_map = Output(layout={'width': '50%'})
output_pie = Output(layout={'width': '50%'})


# --- Fonction de mise à jour (combine la carte ET le camembert) ---
def update_dashboard(departement, show_schools):
    
    # --- 2. LOGIQUE DE LA CARTE ---
    with output_map:
        output_map.clear_output(wait=True)
        
        # Filtrer les GDF pour le département choisi
        dep_equip_gdf = equip_seniors_gdf[equip_seniors_gdf['dep_nom'] == departement]
        dep_edu_gdf = edu_gdf[edu_gdf['Nom_departement'] == departement]
        
        if dep_equip_gdf.empty:
            print(f"Aucun équipement (de la sélection) trouvé pour {departement}")
            return

        # Centrer la carte (méthode simple avec suppression des warnings)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning) # Ignore les UserWarning
            center_lat = dep_equip_gdf.geometry.centroid.y.mean()
            center_lon = dep_equip_gdf.geometry.centroid.x.mean()
        
        m = folium.Map(location=[center_lat, center_lon], zoom_start=9)

        # Ajout des équipements (beaucoup plus rapide car pré-filtré !)
        mc = MarkerCluster(name="Équipements sportifs").add_to(m)
        for _, row in dep_equip_gdf.iterrows():
            folium.Marker(
                location=[row.geometry.y, row.geometry.x],
                popup=f"<b>{row['equip_nom']}</b><br>{row['equip_type_name']}<br>{row['new_name']}",
                icon=folium.Icon(color="blue", icon="fa-person-running", prefix="fa")
            ).add_to(mc)

        # Ajout des écoles si coché
        if show_schools:
            school_layer = MarkerCluster(name="Écoles / Collèges / Lycées").add_to(m)
            for _, row in dep_edu_gdf.iterrows():
                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x],
                    radius=3,
                    color="green", fill=True, fill_opacity=0.7,
                    popup=f"{row['Nom_etablissement']}<br>{row['Nom_commune']}"
                ).add_to(school_layer)
        
        folium.LayerControl().add_to(m)
        display(m)

    # --- 3. LOGIQUE DU CAMEMBERT ---
    with output_pie:
        output_pie.clear_output(wait=True)
        
        # On réutilise les données déjà filtrées
        if dep_equip_gdf.empty:
            print("Pas de données pour le graphique.")
            return

        variety = dep_equip_gdf['equip_type_name'].value_counts()
        
        plt.figure(figsize=(8, 8))
        # Affiche seulement le top 10, regroupe le reste en 'Autres'
        if len(variety) > 10:
            top_10 = variety.head(10)
            others = pd.Series(variety.iloc[10:].sum(), index=['Autres'])
            variety_to_plot = pd.concat([top_10, others])
        else:
            variety_to_plot = variety

        variety_to_plot.plot(
            kind='pie', 
            autopct='%1.1f%%', 
            startangle=90,
            pctdistance=0.85,
            wedgeprops=dict(width=0.4) # Pour un effet "donut"
        )
        plt.title(f"Répartition des activités dans {departement}", pad=20)
        plt.ylabel('') # Cache le label "equip_type_name"
        plt.axis('equal') # Assure un cercle parfait
        plt.show()

# --- Liaison et affichage ---
ui_controls = VBox([dep_widget, school_widget])

dashboard_output = interactive_output(
    update_dashboard, 
    {'departement': dep_widget, 'show_schools': school_widget}
)

dashboard_layout = HBox([output_map, output_pie])

display(ui_controls, dashboard_layout)

Chargement des données...
Équipements sportifs (brut) : 333,204 lignes
Établissements scolaires (brut) : 68,970 lignes
Nettoyage et filtrage...
Équipements sportifs (après filtrage IDF + seniors) : 90,581 lignes
Établissements scolaires (après filtrage IDF) : 68,177 lignes
Préparation terminée.

--- Visualisation 1 : Classement des Départements ---



--- Visualisation 2 & 3 : Dashboard d'Exploration ---
